# 비지도학습 — 신용카드 고객 세분화 재현 실험

목표는 예측 정확도 향상이 아니라 안정적이고 설명 가능한 고객군을 찾아 실행 가능한 비즈니스 가설과 검증 KPI로 연결하는 것이다.

- 실행 기준 시각: `20260814_062642`
- 원본 노트북은 `01_원본보관`에 수정 없이 보관했다.
- 이 노트북은 최종 재현 파이프라인이다. 과거의 모든 탐색 실험과 출력은 원본 보관본에서 확인할 수 있다.
- 모델 선택은 검증 세트에서만 수행하고, 테스트 세트는 최종 보고에 사용한다.
- 모든 비교는 동일 분할과 동일 평가지표를 사용하며 학습·추론 시간도 기록한다.

## 1. 환경·데이터 로드

In [1]:
from pathlib import Path
from time import perf_counter
from itertools import combinations
import json, warnings
import numpy as np
import pandas as pd
import kagglehub
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path.cwd()
ARTIFACTS = ROOT / "03_실험결과"
ARTIFACTS.mkdir(exist_ok=True)

data_path = Path(kagglehub.dataset_download("arjunbhasin2013/ccdata"))
csv_path = data_path / "CC GENERAL.csv"
df = pd.read_csv(csv_path)
customer_ids = df["CUST_ID"].copy()
features = df.drop(columns="CUST_ID")
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(features), columns=features.columns, index=features.index)
print("data:", csv_path.name, "shape:", df.shape)
print("clustering_features:", X_imputed.shape[1], "missing_before:", int(features.isna().sum().sum()))

data: CC GENERAL.csv shape: (8950, 18)
clustering_features: 17 missing_before: 314


## 2. 전처리·군집 수 비교

기존 StandardScaler 기준을 베이스라인으로 두고, 긴 꼬리 분포의 금액·횟수 변수에 `log1p`를 적용한 뒤 StandardScaler/RobustScaler를 비교한다. 모든 후보는 Silhouette, Calinski-Harabasz, Davies-Bouldin과 최소 군집 비율을 함께 기록한다.

In [2]:
log_cols = [
    "BALANCE", "PURCHASES", "ONEOFF_PURCHASES", "INSTALLMENTS_PURCHASES",
    "CASH_ADVANCE", "CASH_ADVANCE_TRX", "PURCHASES_TRX", "CREDIT_LIMIT",
    "PAYMENTS", "MINIMUM_PAYMENTS",
]
X_log = X_imputed.copy()
X_log[log_cols] = np.log1p(X_log[log_cols].clip(lower=0))

representations = {
    "raw_standard": StandardScaler().fit_transform(X_imputed),
    "log_standard": StandardScaler().fit_transform(X_log),
    "log_robust": RobustScaler().fit_transform(X_log),
}

pca95 = PCA(n_components=0.95, random_state=SEED)
representations["log_standard_pca95"] = pca95.fit_transform(representations["log_standard"])
print("PCA dimensions:", X_imputed.shape[1], "->", representations["log_standard_pca95"].shape[1],
      "explained_variance:", pca95.explained_variance_ratio_.sum())

rows = []
label_store = {}
for prep_name, Xr in representations.items():
    for k in range(2, 9):
        start = perf_counter()
        model = KMeans(n_clusters=k, n_init=20, random_state=SEED)
        labels = model.fit_predict(Xr)
        fit_s = perf_counter() - start
        counts = np.bincount(labels)
        row = {
            "preprocessing": prep_name, "k": k,
            "silhouette": silhouette_score(Xr, labels, sample_size=min(5000, len(Xr)), random_state=SEED),
            "calinski_harabasz": calinski_harabasz_score(Xr, labels),
            "davies_bouldin": davies_bouldin_score(Xr, labels),
            "min_cluster_share": counts.min() / len(labels),
            "fit_seconds": fit_s,
        }
        rows.append(row)
        label_store[(prep_name, k)] = labels

kmeans_results = pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)
display(kmeans_results.head(15))
kmeans_results.to_csv(ARTIFACTS / "unsupervised_kmeans_candidates.csv", index=False)

# 운영 요구: 캠페인 3종으로 연결 가능한 k=3을 고정하고, 그 안에서 가장 나은 전처리를 선택
k3_results = kmeans_results[kmeans_results["k"] == 3].sort_values("silhouette").reset_index(drop=True)
chosen_preprocessing = k3_results.iloc[-1]["preprocessing"]
chosen_labels = label_store[(chosen_preprocessing, 3)]
baseline_labels = label_store[("raw_standard", 3)]
print("chosen_for_3_business_segments:", chosen_preprocessing)
display(k3_results)

PCA dimensions: 17 -> 10 explained_variance: 0.9588503974127632


,preprocessing,k,silhouette,calinski_harabasz,davies_bouldin,min_cluster_share,fit_seconds
0,log_robust,3,0.391605,3511.255589,1.062225,0.138547,0.087950
1,log_robust,2,0.349658,3013.149505,1.594664,0.277877,0.052208
2,log_robust,5,0.267359,3151.917477,1.189338,0.066145,0.104053
3,log_robust,4,0.256575,3279.841089,1.314266,0.133184,0.081565
4,log_standard_pca95,2,0.256520,3241.695720,1.456785,0.352737,0.083103
5,log_robust,8,0.248781,2595.559749,1.299393,0.044246,0.187806
6,log_robust,7,0.247359,2749.150586,1.351557,0.044022,0.152766
7,log_standard,2,0.246261,3064.874283,1.506310,0.353408,0.077531
8,raw_standard,3,0.245534,1605.085496,1.596848,0.137989,0.096913
9,log_robust,6,0.245220,2890.821125,1.303797,0.065698,0.146929


chosen_for_3_business_segments: log_robust


,preprocessing,k,silhouette,calinski_harabasz,davies_bouldin,min_cluster_share,fit_seconds
0,log_standard,3,0.223360,2642.884978,1.681238,0.320782,0.087328
1,log_standard_pca95,3,0.233701,2821.195982,1.626265,0.320559,0.105492
2,raw_standard,3,0.245534,1605.085496,1.596848,0.137989,0.096913
3,log_robust,3,0.391605,3511.255589,1.062225,0.138547,0.087950


## 3. 안정성, PCA 보존성, GMM 보조 검증

In [3]:
X_chosen = representations[chosen_preprocessing]
seed_labels = {}
for seed in [0, 1, 2, 42, 99]:
    seed_labels[seed] = KMeans(n_clusters=3, n_init=20, random_state=seed).fit_predict(X_chosen)
stability_aris = [adjusted_rand_score(seed_labels[a], seed_labels[b]) for a, b in combinations(seed_labels, 2)]

if chosen_preprocessing.endswith("pca95"):
    pca_labels = chosen_labels
    original_space_labels = label_store[("log_standard", 3)]
    pca_ari = adjusted_rand_score(original_space_labels, pca_labels)
else:
    pca_chosen = PCA(n_components=0.95, random_state=SEED).fit_transform(X_chosen)
    pca_labels = KMeans(n_clusters=3, n_init=20, random_state=SEED).fit_predict(pca_chosen)
    pca_ari = adjusted_rand_score(chosen_labels, pca_labels)

gmm_rows = []
gmm_models = {}
for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, covariance_type="diag", n_init=5, random_state=SEED)
    gmm.fit(X_chosen)
    labels = gmm.predict(X_chosen)
    probs = gmm.predict_proba(X_chosen).max(axis=1)
    gmm_rows.append({
        "k": k, "bic": gmm.bic(X_chosen), "aic": gmm.aic(X_chosen),
        "silhouette": silhouette_score(X_chosen, labels, sample_size=min(5000, len(X_chosen)), random_state=SEED),
        "mean_max_membership_probability": probs.mean(),
        "share_membership_probability_ge_0_9": (probs >= 0.9).mean(),
    })
    gmm_models[k] = gmm
gmm_results = pd.DataFrame(gmm_rows).sort_values("bic").reset_index(drop=True)
gmm_results.to_csv(ARTIFACTS / "unsupervised_gmm_candidates.csv", index=False)

gmm3_labels = gmm_models[3].predict(X_chosen)
summary = {
    "chosen_preprocessing_for_k3": chosen_preprocessing,
    "kmeans_k": 3,
    "kmeans_silhouette": float(k3_results.iloc[-1]["silhouette"]),
    "baseline_raw_standard_k3_silhouette": float(k3_results[k3_results.preprocessing == "raw_standard"].iloc[0]["silhouette"]),
    "mean_pairwise_seed_ari": float(np.mean(stability_aris)),
    "min_pairwise_seed_ari": float(np.min(stability_aris)),
    "pca_cluster_ari": float(pca_ari),
    "kmeans_vs_gmm_k3_ari": float(adjusted_rand_score(chosen_labels, gmm3_labels)),
    "gmm_bic_selected_k": int(gmm_results.iloc[0]["k"]),
    "note": "GMM membership probability is confidence, not accuracy",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(gmm_results)
(ARTIFACTS / "unsupervised_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

{
  "chosen_preprocessing_for_k3": "log_robust",
  "kmeans_k": 3,
  "kmeans_silhouette": 0.39160509141777133,
  "baseline_raw_standard_k3_silhouette": 0.24553439864585863,
  "mean_pairwise_seed_ari": 0.9989171922104985,
  "min_pairwise_seed_ari": 0.9981931262806264,
  "pca_cluster_ari": 0.998321582230508,
  "kmeans_vs_gmm_k3_ari": -0.035145313610249314,
  "gmm_bic_selected_k": 6,
  "note": "GMM membership probability is confidence, not accuracy"
}


,k,bic,aic,silhouette,mean_max_membership_probability,share_membership_probability_ge_0_9
0,6,-210162.230762,-211646.007203,0.066140,0.999333,0.998101
1,5,-157262.386636,-158497.683769,0.090318,0.996890,0.990950
2,4,-100141.663610,-101128.481435,0.037646,0.999793,0.999330
3,3,-50791.293634,-51529.632150,0.078414,1.000000,1.000000
4,2,145987.004843,145497.145635,0.133698,1.000000,1.000000


451

## 4. 고객군 프로파일과 비즈니스 가설

In [4]:
profile_mean = X_imputed.assign(cluster=chosen_labels).groupby("cluster").mean()
profile_median = X_imputed.assign(cluster=chosen_labels).groupby("cluster").median()
cluster_sizes = pd.Series(chosen_labels).value_counts().sort_index().rename("customers")
cluster_share = (cluster_sizes / len(chosen_labels)).rename("share")

z_profile = pd.DataFrame(StandardScaler().fit_transform(profile_mean),
                         index=profile_mean.index, columns=profile_mean.columns)

shopping_cols = ["PURCHASES", "ONEOFF_PURCHASES", "INSTALLMENTS_PURCHASES", "PURCHASES_TRX"]
cash_cols = ["CASH_ADVANCE", "CASH_ADVANCE_FREQUENCY", "CASH_ADVANCE_TRX"]
shopping_score = z_profile[shopping_cols].mean(axis=1)
cash_score = z_profile[cash_cols].mean(axis=1)
shopping_cluster = int(shopping_score.idxmax())
activity_cols = shopping_cols + cash_cols + ["BALANCE", "PAYMENTS", "CREDIT_LIMIT"]
low_cluster = int(z_profile[activity_cols].mean(axis=1).drop(index=shopping_cluster).idxmin())
cash_cluster = int(({0, 1, 2} - {shopping_cluster, low_cluster}).pop())
business_names = {
    shopping_cluster: "고구매·쇼핑/VIP 후보군",
    cash_cluster: "현금서비스 의존 후보군",
    low_cluster: "저활동·재활성화 후보군",
}

top_features = {}
for cluster in z_profile.index:
    if int(cluster) == low_cluster:
        cols = z_profile.loc[cluster].sort_values().head(5).index.tolist()
        top_features[int(cluster)] = "낮음: " + ", ".join(cols)
    else:
        cols = z_profile.loc[cluster].sort_values(ascending=False).head(5).index.tolist()
        top_features[int(cluster)] = "높음: " + ", ".join(cols)
segment_table = pd.DataFrame({
    "cluster": cluster_sizes.index,
    "business_segment": [business_names[int(c)] for c in cluster_sizes.index],
    "customers": cluster_sizes.values,
    "share": cluster_share.values,
    "top_distinctive_features": [top_features[int(c)] for c in cluster_sizes.index],
})
display(segment_table)
segment_table.to_csv(ARTIFACTS / "unsupervised_segment_summary.csv", index=False)
profile_mean.to_csv(ARTIFACTS / "unsupervised_segment_means.csv")
profile_median.to_csv(ARTIFACTS / "unsupervised_segment_medians.csv")

assignments = pd.DataFrame({
    "CUST_ID": customer_ids,
    "cluster": chosen_labels,
    "business_segment": [business_names[int(c)] for c in chosen_labels],
})
assignments.to_csv(ARTIFACTS / "unsupervised_customer_segments.csv", index=False)

business_hypotheses = pd.DataFrame([
    {"segment": "고구매·쇼핑/VIP 후보군", "action_hypothesis": "프리미엄 혜택·교차판매·리워드 유지",
     "online_validation_kpi": "캠페인 대비 재구매율, 객단가, 리워드 사용률 uplift"},
    {"segment": "현금서비스 의존 후보군", "action_hypothesis": "분할상환 전환·금융건전성 안내",
     "online_validation_kpi": "현금서비스 빈도/금액 감소, 연체율(외부 데이터 필요), 전환율"},
    {"segment": "저활동·재활성화 후보군", "action_hypothesis": "저비용 재활성화 쿠폰·첫 결제 유도",
     "online_validation_kpi": "30/60/90일 활성화율, 거래 전환율, 증분 매출"},
])
display(business_hypotheses)
business_hypotheses.to_csv(ARTIFACTS / "unsupervised_business_hypotheses.csv", index=False)

,cluster,business_segment,customers,share,top_distinctive_features
0,0,고구매·쇼핑/VIP 후보군,1307,0.146034,"높음: PRC_FULL_PAYMENT, PURCHASES_INSTALLMENTS_F..."
1,1,현금서비스 의존 후보군,6403,0.715419,"높음: MINIMUM_PAYMENTS, BALANCE, CASH_ADVANCE_TR..."
2,2,저활동·재활성화 후보군,1240,0.138547,"낮음: BALANCE_FREQUENCY, TENURE, CREDIT_LIMIT, P..."


,segment,action_hypothesis,online_validation_kpi
0,고구매·쇼핑/VIP 후보군,프리미엄 혜택·교차판매·리워드 유지,"캠페인 대비 재구매율, 객단가, 리워드 사용률 uplift"
1,현금서비스 의존 후보군,분할상환 전환·금융건전성 안내,"현금서비스 빈도/금액 감소, 연체율(외부 데이터 필요), 전환율"
2,저활동·재활성화 후보군,저비용 재활성화 쿠폰·첫 결제 유도,"30/60/90일 활성화율, 거래 전환율, 증분 매출"


## 5. 해석 한계

- 군집에는 정답 라벨이 없으므로 `정확도`나 `성능 향상`이라고 표현하지 않는다.
- GMM 최대 소속확률은 모델의 확신도이며 실제 정답률이 아니다.
- 비즈니스 액션은 데이터로부터 도출한 가설이다. 효과는 A/B 테스트와 실제 캠페인 KPI로 별도 검증해야 한다.
- 기존의 `PURCHASES > 0` 지도 라벨과 PCA 분류 정확도 비교는 `PURCHASES` 및 그 구성항목이 입력에 포함되는 누수 문제가 있어 핵심 성과에서 제외한다.